# 🔤 Devnagri — Multilingual Transliteration Model Training

This notebook trains a character-level Transformer model for English → Hindi/Bengali/Tamil transliteration using OpenNMT-py on Google Colab's free GPU.

**Steps:**
1. Install dependencies
2. Mount Google Drive & upload data
3. Create training configuration
4. Build vocabulary
5. Train the model (checkpoints saved to Google Drive)
6. Evaluate on test set
7. Convert to CTranslate2
8. Download trained models

## 1. Setup & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install OpenNMT-py and dependencies
!pip install OpenNMT-py>=3.0 editdistance ctranslate2

# Fix NumPy version (torch 2.2.x needs numpy<2)
!pip install "numpy<2"

In [ ]:
# Verify installations
import torch
import onmt
import ctranslate2
import editdistance

print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"OpenNMT-py:   {onmt.__version__}")
print(f"CTranslate2:  {ctranslate2.__version__}")
print("✅ All good!")

## 2. Mount Google Drive & Upload Data

⚠️ **Important:** We mount Google Drive to save checkpoints there. This prevents losing trained models if the Colab session disconnects.

Run `python data/download_data.py` and `python data/preprocess.py` locally first, then upload the `data/processed/` files.

In [ ]:
# Mount Google Drive (REQUIRED — saves checkpoints safely)
from google.colab import drive
drive.mount('/content/drive')

# Create Google Drive directories for model storage
!mkdir -p /content/drive/MyDrive/devnagri/models
!mkdir -p /content/drive/MyDrive/devnagri/results
print('Google Drive mounted and directories created ✅')

In [ ]:
# Create local directories
!mkdir -p models configs results data/processed

In [ ]:
# Upload processed data files
import os
from google.colab import files

print('Upload: src-train.txt, tgt-train.txt, src-val.txt, tgt-val.txt, src-test.txt, tgt-test.txt')
uploaded = files.upload()

for filename, content in uploaded.items():
    filepath = f'data/processed/{filename}'
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f'Saved: {filepath}')

In [ ]:
# Verify data files
!echo '--- Data Files ---'
!ls -la data/processed/
!echo ''
!echo '--- Line Counts ---'
!wc -l data/processed/*.txt
!echo ''
!echo '--- Sample Lines (src-train) ---'
!head -5 data/processed/src-train.txt
!echo ''
!echo '--- Sample Lines (tgt-train) ---'
!head -5 data/processed/tgt-train.txt

## 3. Create Training Configuration

**Note:** Checkpoints are saved to Google Drive (`/content/drive/MyDrive/devnagri/models/`) so they survive session disconnects.

In [ ]:
%%writefile configs/transliteration.yaml

## Model and Embeddings
model_type: text
encoder_type: transformer
decoder_type: transformer

# Transformer parameters (small, suitable for character-level transliteration)
enc_layers: 4
dec_layers: 4
heads: 4
hidden_size: 256
transformer_ff: 1024
dropout: [0.1]
attention_dropout: [0.1]

# Embeddings
src_word_vec_size: 256
tgt_word_vec_size: 256
position_encoding: true
share_embeddings: true

## Data
data:
  corpus_1:
    path_src: data/processed/src-train.txt
    path_tgt: data/processed/tgt-train.txt
    transforms: [filtertoolong]
    weight: 1
  valid:
    path_src: data/processed/src-val.txt
    path_tgt: data/processed/tgt-val.txt
    transforms: [filtertoolong]

# Vocabulary
src_vocab: models/vocab.src
tgt_vocab: models/vocab.tgt
share_vocab: true
src_vocab_size: 5000
tgt_vocab_size: 5000

# Filtering
src_seq_length: 150
tgt_seq_length: 150

## Training
batch_size: 4096
batch_type: tokens
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

optim: adam
adam_beta1: 0.9
adam_beta2: 0.998
learning_rate: 2.0
warmup_steps: 4000
decay_method: noam
label_smoothing: 0.1
max_grad_norm: 0

train_steps: 30000
valid_steps: 1000
save_checkpoint_steps: 5000
keep_checkpoint: 5

report_every: 100
log_file: /content/drive/MyDrive/devnagri/models/train.log
save_model: /content/drive/MyDrive/devnagri/models/transliteration_model

world_size: 1
gpu_ranks: [0]

## 4. Build Vocabulary

In [ ]:
!python -m onmt.bin.build_vocab \
    -config configs/transliteration.yaml \
    -save_data models/data \
    -n_sample -1

In [ ]:
# Check vocabulary
!echo '--- Vocabulary Files ---'
!ls -la models/vocab.*
!echo ''
!echo '--- Vocab Size ---'
!wc -l models/vocab.*

## 5. Train the Model

Training ~30K steps on Colab T4 GPU. Estimated time: **~3 hours**.

Checkpoints saved every 5000 steps **to Google Drive** (safe from disconnects).

In [ ]:
# Full training (30K steps) — checkpoints saved to Google Drive
!python -m onmt.bin.train \
    -config configs/transliteration.yaml \
    -save_data models/data \
    -world_size 1 \
    -gpu_ranks 0

In [ ]:
# Check saved checkpoints (on Google Drive)
!ls -la /content/drive/MyDrive/devnagri/models/transliteration_model_step_*.pt

## 6. Evaluate the Model

In [ ]:
import glob

# Find the latest checkpoint (from Google Drive)
checkpoints = sorted(glob.glob('/content/drive/MyDrive/devnagri/models/transliteration_model_step_*.pt'))
latest = checkpoints[-1] if checkpoints else None
print(f'Latest checkpoint: {latest}')

In [ ]:
# Run translation on test set
!python -m onmt.bin.translate \
    -model $latest \
    -src data/processed/src-test.txt \
    -output results/predictions.txt \
    -beam_size 5 \
    -replace_unk \
    -gpu 0

In [ ]:
# Compute evaluation metrics
import editdistance
from collections import defaultdict
import json

def chars_to_word(s):
    return s.replace(' ', '').strip()

def get_lang(s):
    if s.startswith('<') and '>' in s:
        return s[1:s.index('>')]
    return 'unknown'

with open('data/processed/src-test.txt') as f:
    sources = [l.strip() for l in f]
with open('data/processed/tgt-test.txt') as f:
    references = [l.strip() for l in f]
with open('results/predictions.txt') as f:
    predictions = [l.strip() for l in f]

per_lang = defaultdict(lambda: {'total': 0, 'correct': 0, 'cer_dist': 0, 'cer_len': 0})

for src, ref, pred in zip(sources, references, predictions):
    lang = get_lang(src)
    p = chars_to_word(pred)
    r = chars_to_word(ref)
    s = per_lang[lang]
    s['total'] += 1
    if p == r: s['correct'] += 1
    s['cer_dist'] += editdistance.eval(list(p), list(r))
    s['cer_len'] += len(r)

print(f"{'Lang':<10} {'Pairs':>8} {'Acc%':>10} {'CER%':>10}")
print('-' * 40)
for lang, s in sorted(per_lang.items()):
    acc = s['correct']/s['total']*100
    cer = s['cer_dist']/s['cer_len']*100
    print(f"{lang:<10} {s['total']:>8} {acc:>9.2f} {cer:>9.2f}")

## 7. Convert to CTranslate2

In [ ]:
# Convert to CTranslate2 with int8 quantization
!ct2-opennmt-py-converter \
    --model_path $latest \
    --output_dir models/ct2_model \
    --quantization int8

In [ ]:
# Quick CTranslate2 inference test
import ctranslate2

translator = ctranslate2.Translator('models/ct2_model', device='cpu')

test_words = [
    ('<hin>', 'namaste'),
    ('<ben>', 'kolkata'),
    ('<tam>', 'chennai'),
]

for prefix, word in test_words:
    tokens = [prefix] + list(word)
    result = translator.translate_batch([tokens], beam_size=5)
    output = ''.join(result[0].hypotheses[0])
    print(f"{word} → {output}")

## 8. Save & Download Trained Models

Checkpoints are already on Google Drive. Now save the CT2 model and results there too.

In [ ]:
# Save CT2 model and results to Google Drive
!cp -r models/ct2_model /content/drive/MyDrive/devnagri/models/
!cp -r results /content/drive/MyDrive/devnagri/
print('✅ CT2 model and results saved to Google Drive!')

In [ ]:
# Download zip files to local machine
!zip -r models_download.zip models/ct2_model/ 
!zip -r results_download.zip results/

from google.colab import files
files.download('models_download.zip')
files.download('results_download.zip')

## 🔄 Resume Training (if session disconnected)

If your Colab session disconnected during training, run the cells above to reinstall packages and re-upload data, then use this cell to resume from the last checkpoint.

In [ ]:
# # Resume training from last checkpoint (uncomment to use)
# import glob
# checkpoints = sorted(glob.glob('/content/drive/MyDrive/devnagri/models/transliteration_model_step_*.pt'))
# resume_from = checkpoints[-1] if checkpoints else None
# print(f'Resuming from: {resume_from}')
#
# !python -m onmt.bin.train \
#     -config configs/transliteration.yaml \
#     -save_data models/data \
#     -train_from $resume_from \
#     -world_size 1 \
#     -gpu_ranks 0